# 00 — The real NOSIBLE API, as a reference point

Blog: [The Road to Cybernaut-1](https://nosible.com/blog/the-road-to-cybernaut-1)

This notebook is **not** part of `cybernaut-mini`. It calls the real, hosted NOSIBLE
search API. It is kept because the shape of that API is itself evidence for what the
blog describes — and it gives us a ground truth to compare our replica against.

## Why this matters for the replica

Look at the request body below. It carries an **`instruction`** field alongside the
**`question`**. That is not decoration — it is the public surface of blog **Stage 4,
"Instruction Tuning and Embedding"**:

> "we generate an appropriate instruction for the question and submit it, along with
> the expansions, to `multilingual-e5-large-instruct`. E5 is an open-source,
> instruction-tuned embedding model from Microsoft Research. It is designed to align
> vectors with natural language instructions."

The blog claims optimizing this instruction yields "a free 1-5% improvement in search
precision and recall". The API exposing `instruction` as a caller-controlled parameter
is consistent with that claim, and is why our replica treats the instruction template
as a **tunable knob** rather than a constant (see `query/s4_embed/`).

Note also `n_results: 10` — and recall the blog's design principle #2, *recall over
precision*: "People need precision; AIs need recall." The worked examples in the post
return **100** results.

## Credentials

This notebook reads `NOSIBLE_API_KEY` from the environment. **Never paste a key into a
notebook cell** — notebooks serialize their source to disk and get committed. An
earlier version of this file had a live key hardcoded in three cells.

```bash
export NOSIBLE_API_KEY="nos_sk_..."
```

If the variable is unset, every cell below degrades to a no-op with an explanatory
message rather than failing — this notebook is documentation first, and must remain
readable without an account.

In [ ]:
import json
import os

API_KEY = os.environ.get("NOSIBLE_API_KEY")
ENDPOINT = "https://www.nosible.ai/search/v1/fast-search"

if API_KEY:
    print(f"NOSIBLE_API_KEY found (ends ...{API_KEY[-4:]})")
else:
    print("NOSIBLE_API_KEY is not set — live cells below will be skipped.")
    print("This notebook is still readable as documentation without it.")

## The request shape

Shown as data rather than executed, so the structure is legible even offline.
Each field is annotated with the blog stage it corresponds to.

In [ ]:
REQUEST_SHAPE = {
    "page": {
        # Stage 4 — Instruction Tuning and Embedding.
        # The instruction is prepended to the query before embedding, steering the
        # vector. This is the knob the blog says is worth 1-5% precision/recall.
        "instruction": "Retrieve semantically similar text.",
        # Stages 1-3 — the raw question, which the engine detects the language of,
        # tokenizes, and extracts search intents from.
        "question": "Why did Argus Filch confiscate the Marauder's Map?",
        # Stage 8 — how many results to reduce to. The blog's design principle is
        # "recall over precision": AIs have massive context windows, so ask for many.
        "n_results": 10,
    }
}

print(json.dumps(REQUEST_SHAPE, indent=2))

In [ ]:
def fast_search(question: str, instruction: str, n_results: int = 10) -> dict | None:
    """Call the hosted NOSIBLE API. Returns None when no key is configured."""
    if not API_KEY:
        print("Skipped: NOSIBLE_API_KEY is not set.")
        return None

    import requests

    response = requests.post(
        ENDPOINT,
        headers={"Content-Type": "application/json", "api_key": API_KEY},
        json={
            "page": {
                "instruction": instruction,
                "question": question,
                "n_results": n_results,
            }
        },
        timeout=30,
    )
    response.raise_for_status()
    return response.json()


result = fast_search(
    question="Why did Argus Filch confiscate the Marauder's Map?",
    instruction="Retrieve semantically similar text.",
)

if result:
    for hit in result["response"]:
        print(hit["title"])

## Experiment: does the instruction actually change the results?

This is the blog's Stage 4 claim, testable directly against the live API. If the
instruction is a real parameter and not a no-op, two different instructions over the
*same* question should return measurably different result sets.

We run the blog's own worked-example query so the comparison is on their turf.

In [ ]:
BLOG_QUESTION = (
    "What lessons from bacteria and yeast actually translate into "
    "safer gene-editing medicines?"
)

# A generic instruction vs. the evolved template quoted in the blog.
INSTRUCTIONS = {
    "generic": "Retrieve semantically similar text.",
    "blog_evolved": (
        "Given a question, please retrieve any relevant English Headlines, Leads, "
        "Passages, and Source URLs that focus on the same named entities as the "
        "question, and provide substantive answers to the question."
    ),
}

titles: dict[str, list[str]] = {}
for name, instruction in INSTRUCTIONS.items():
    payload = fast_search(BLOG_QUESTION, instruction, n_results=10)
    if payload is None:
        break
    titles[name] = [hit["title"] for hit in payload["response"]]
    print(f"\n--- {name} ---")
    for title in titles[name]:
        print(" ", title)

if len(titles) == 2:
    a, b = (set(v) for v in titles.values())
    overlap = len(a & b) / len(a | b) if (a | b) else 0.0
    print(f"\nJaccard overlap between the two result sets: {overlap:.2f}")
    print(
        "Overlap well below 1.0 means the instruction genuinely steers retrieval, "
        "which is what blog Stage 4 claims."
    )

## What this tells us to build

| Observation from the live API | Consequence for `cybernaut-mini` |
|---|---|
| `instruction` is a caller-supplied parameter | Stage 4 needs a **template registry**, not a hardcoded string — `query/s4_embed/` |
| `question` is passed raw | Language detection, tokenization and intent extraction happen **server-side** — stages 1-3 are ours to build |
| `n_results` defaults generously | Reduce phase must scale to ~100 hits, not 10 — blog principle "recall over precision" |
| Response items carry a `title` | Snippets and titles are the product surface — see Stage 8 snippet construction |

**Alternatives to calling this API at all:** we could have treated the blog as the
sole specification and never touched the hosted service. We keep this notebook because
a reachable reference implementation is a cheap oracle — when our replica behaves
oddly, it is useful to know how the real thing responds to the same input. It is *not*
used for evaluation: our metrics come from real human relevance judgments (SciFact
qrels), not from agreement with a black box.